# **Preparing Rainfall Data**
*Notebook created Jan. 22, 2026*

This notebook documents the processing of PAGASA station rainfall data in preparation for evaluating the relationship between PH-SWM and ISM. PHL rainfall data will be processed based on the methodology devised by [Cruz et al. (2013)](https://doi.org/10.1016/j.atmosres.2012.06.010). All processes will be performed locally.

## **1. Downloading Datasets**
The following datasets will be used in comparing Indian and Philippine rainfall.

| **Dataset** | **Source** | **Period** | **Temporal Resolution** |
| :--- | :---: | :---: | :---: |
| India | [India Meteorological Department](https://dsp.imdpune.gov.in/home_ogd_rainfall.php) | 1901 – present | Monthly |
| Philippines | [DOST-PAGASA](https://www.pagasa.dost.gov.ph/) | 1979 – present | Daily |

## **2. Preparing PHL Rainfall**
**Given:** All-India rainfall are spatially averaged and resolved monthly. Representative Philippine rainfall are station-based and resolve daily; additionally, only 11 stations on the western region of the Philippines are selected (less than 20% data missing).
- PAGASA stations: Ambulong, Baguio, Coron, Cuyo, Dagupan, Iba, Iloilo, Laoag, Port Area, Sangley Point, Science Garden

**Objective:** Generate a .csv file of monthly and spatially averaged Philippine rainfall data.

### **A. Subset to JJAS**
PHL rainfall will be subset to Jun-Sep season (JJAS). New .csv files will be generated and put into `Data/[folder]` folder.

### **B. Monthly Accumulated Rainfall**
Monthly accumulated rainfall will be computed using `mo_ave.py`. If the station has missing or trace data in its daily rainfall data, the monthly climatological mean will be used instead.

Climatological normals are extracted using [PAGASAClimateNormals-pdf2txt2csv](https://github.com/LuisSleepy/PAGASAClimateNormals-pdf2txt2csv) by Jan Luis Antoc.

### **C. Monthly Average Rainfall**
Monthly average rainfall for western Philippines is computed as the mean of the PAGASA stations selected for this study. The western region of the Philippines is most affected by the southwest monsoon every year.

In [ ]:
import pandas as pd
import numpy as np

stations = [
    "Ambulong", "Baguio", "Coron", "Cuyo", "Dagupan",
    "Iba", "Iloilo", "Laoag", "Port Area", "Sangley Point", "Science Garden"
]
clim_file = "climnormals.csv"
output_file = "mean_monthly_rainfall.csv"
months = {
    6: "jun",
    7: "jul",
    8: "aug",
    9: "sep"
}

clim = pd.read_csv(clim_file)
clim.columns = clim.columns.str.strip().str.upper()
clim.rename(columns={"STATION": "station"}, inplace=True)
clim.set_index("station", inplace=True)

station_monthly = {}

for st in stations:
    df = pd.read_csv(f"{st} Daily Data.csv")

    df.columns = df.columns.str.strip().str.lower()
    df["date"] = pd.to_datetime(
        df[["year", "month", "day"]]
    )
    df.set_index("date", inplace=True)

    records = {}

    for yr in df["year"].unique():
        records[yr] = {}
        for m in months:
            data = df[
                (df.index.year == yr) &
                (df.index.month == m)
            ]["rainfall"]

            if data.empty or data.isin([-999, -1]).any():
                records[yr][m] = clim.loc[st, months[m].upper()]
            else:
                records[yr][m] = data.sum()

    station_monthly[st] = pd.DataFrame.from_dict(records, orient="index")

all_years = sorted(
    set().union(*[df.index for df in station_monthly.values()])
)

output = []

for yr in all_years:
    row = {"year": yr}

    for m, name in months.items():
        values = [
            station_monthly[st].loc[yr, m]
            for st in stations
            if yr in station_monthly[st].index
        ]

        row[name] = np.mean(values)

    row["annual"] = np.mean([row[m] for m in months.values()])
    output.append(row)

final_df = pd.DataFrame(output)
final_df.to_csv(output_file, index=False)

print("✔ Mean monthly rainfall saved to:", output_file)